# Reservoir Data Loader for Kaggle

Loads WRIS reservoir data from your uploaded Kaggle dataset.  
If no dataset is attached, generates synthetic data as fallback.

### How to get data here
1. **Locally:** `python scripts/download_wris_data.py` (auto-downloads from NWIC/WRIS)
2. **Locally:** `python scripts/download_wris_data.py --push-kaggle YOUR_USERNAME`
3. **Here:** Add dataset `YOUR_USERNAME/wris-reservoir-data` as input

Or just run this notebook â€” it falls back to synthetic data if nothing is attached.

In [ ]:
# â”€â”€â”€ Config â”€â”€â”€
import os, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd

RESERVOIRS = {
    "nagarjuna_sagar": "Nagarjuna Sagar",
    "srisailam": "Srisailam",
    "almatti": "Almatti",
    "tungabhadra": "Tungabhadra",
    "ujjani": "Ujjani",
    "mettur": "Mettur",
    "krishnaraja_sagara": "Krishnaraja Sagara",
    "jayakwadi": "Jayakwadi",
    "sardar_sarovar": "Sardar Sarovar",
    "ukai": "Ukai",
}

# Possible Kaggle input directories
KAGGLE_DIRS = [
    Path("/kaggle/input/wris-reservoir-data"),
    Path("/kaggle/input/india-reservoir-data"),
    Path("/kaggle/input/cwc-reservoir-data"),
]
WORKING_DIR = Path("/kaggle/working/data/raw/wris")

In [ ]:
# â”€â”€â”€ Source 1: Try Kaggle dataset input â”€â”€â”€

reservoirs = {}
source_used = None

for kdir in KAGGLE_DIRS:
    if kdir.exists():
        csvs = sorted(kdir.glob("*.csv"))
        if csvs:
            source_used = str(kdir)
            for f in csvs:
                rid = f.stem
                if rid in RESERVOIRS:
                    reservoirs[rid] = pd.read_csv(f, parse_dates=["Date"])
            break

if reservoirs:
    print(f"âœ“ Loaded {len(reservoirs)} reservoirs from Kaggle input: {source_used}")
else:
    print("âš  No Kaggle dataset found. Will generate synthetic data in next cell.")

In [ ]:
# â”€â”€â”€ Source 2: Synthetic fallback â”€â”€â”€

if not reservoirs:
    raise RuntimeError("Failed to download data from Kaggle or API. Real data is required.")


In [ ]:
# â”€â”€â”€ Copy to working directory â”€â”€â”€

WORKING_DIR.mkdir(parents=True, exist_ok=True)
for rid, df in reservoirs.items():
    df.to_csv(WORKING_DIR / f"{rid}.csv", index=False)

# Summary
summary = pd.DataFrame([
    {"Reservoir": RESERVOIRS.get(rid, rid), "Rows": len(df),
     "From": str(df["Date"].min())[:10], "To": str(df["Date"].max())[:10],
     "Cols": ", ".join(df.columns)}
    for rid, df in sorted(reservoirs.items())
])

print(f"\nSource: {source_used}")
print(f"Total: {summary['Rows'].sum():,} rows across {len(reservoirs)} reservoirs")
print(f"Saved to: {WORKING_DIR}")
display(summary)

In [ ]:
# â”€â”€â”€ Sanity plot â”€â”€â”€

import matplotlib.pyplot as plt

ncols = 5
nrows = (len(reservoirs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 3 * nrows), sharex=True)
axes = axes.flatten()

for ax, (rid, df) in zip(axes, sorted(reservoirs.items())):
    dates = pd.to_datetime(df["Date"]) if df["Date"].dtype == "object" else df["Date"]
    if "storage" in df.columns:
        ax.plot(dates, df["storage"], linewidth=0.5, color="steelblue")
    ax.set_title(RESERVOIRS.get(rid, rid), fontsize=9)
    ax.tick_params(labelsize=7)

# Hide empty subplots
for ax in axes[len(reservoirs):]:
    ax.set_visible(False)

fig.suptitle(f"Daily Storage â€” {source_used}", fontsize=13)
plt.tight_layout()
plt.show()